In [39]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [40]:
from dotenv import load_dotenv
load_dotenv()

True

In [41]:
from openai import OpenAI
import os

# Initialize client using the variables injected from your .env
openai_client = OpenAI(
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)


In [42]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = response = openai_client.chat.completions.create(
        model='gpt-4o',  # An actual model string available on GitHub
        messages=messages)

response.choices[0].message.content

"Of course! Whether you can join a course typically depends on a few factors, such as its enrollment deadlines, availability, and prerequisites. If the course you're interested in is still accepting participants, go ahead and sign up. If it has already started, it's worth reaching out to the instructor or support team to see if late enrollment is possible. Some courses, especially online ones, offer rolling admissions or allow you to catch up at your own pace.\n\nIf you need help with specifics (e.g., enrollment process, deadlines, etc.), feel free to provide more details about the course, and I’d be happy to assist!"

In [43]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

index = build_index(documents)

In [44]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [45]:

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Searches the web for information",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
	     "additionalProperties": False
        }
    }
    }

In [46]:
response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=[search_tool]
)

response.choices[0].message.content

In [47]:
response.choices

[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BreRk5z0bXspJhSz9UXgq7YD', function=Function(arguments='{"query":"Can I join the course after it has started?"}', name='search'), type='function')]), content_filter_results={})]

In [48]:
print(response)

ChatCompletion(id='chatcmpl-DpsgsZ99Htno8lIE10UtpVXJb8g7n', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BreRk5z0bXspJhSz9UXgq7YD', function=Function(arguments='{"query":"Can I join the course after it has started?"}', name='search'), type='function')]), content_filter_results={})], created=1781257330, model='gpt-4o-2024-11-20', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_49e2bef596', usage=CompletionUsage(completion_tokens=23, prompt_tokens=65, total_tokens=88, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0), latency_checkpoint={'engine_tbt_ms': 8, 'engine_ttft_ms': 

In [49]:
import json

call = response.choices[0].message.tool_calls[0]
args = json.loads(call.function.arguments)
call


ChatCompletionMessageFunctionToolCall(id='call_BreRk5z0bXspJhSz9UXgq7YD', function=Function(arguments='{"query":"Can I join the course after it has started?"}', name='search'), type='function')

In [50]:
    
call.function.name

'search'

In [51]:
results = search(**args)



In [ ]:

result_json = json.dumps(results, indent=2)

In [53]:
function_call_output = {
    "type" : "function_call_output",
    "call_id" : call.id,
    "output" : result_json
}
call.id

'call_BreRk5z0bXspJhSz9UXgq7YD'

In [54]:
messages.append({
    "role": "assistant",
    "content": response.choices[0].message.content,
    "tool_calls": response.choices[0].message.tool_calls
})


messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_BreRk5z0bXspJhSz9UXgq7YD', function=Function(arguments='{"query":"Can I join the course after it has started?"}', name='search'), type='function')]}]

In [55]:
messages.append({
    "role": "tool",
    "tool_call_id": call.id,
    "content": result_json
})

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_BreRk5z0bXspJhSz9UXgq7YD', function=Function(arguments='{"query":"Can I join the course after it has started?"}', name='search'), type='function')]},
 {'role': 'tool',
  'tool_call_id': 'call_BreRk5z0bXspJhSz9UXgq7YD',
  'content': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get 

In [67]:
response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=[search_tool]
)




BadRequestError: Error code: 400 - {'error': {'message': "Missing required parameter: 'messages[3].role'.", 'type': 'invalid_request_error', 'param': 'messages[3].role', 'code': 'missing_required_parameter'}}

In [58]:
response.choices[0].message.content

'Yes, you can still join the course, even if it has already started. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [59]:
def make_call(call):
    args = json.loads(call.function.arguments)

    if call.function.name =="search":
        results = search(**args)

    result_json = json.dumps(results, indent=2)

    return {
    "type" : "tool",
    "tool_call_id" : call.id,
    "output" : result_json
}

In [62]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'system', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [63]:
response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=[search_tool]
)


In [64]:
message = response.choices[0].message

# Append the assistant's turn
messages.append({
    "role": "assistant",
    "content": message.content,
    "tool_calls": message.tool_calls
})

if message.tool_calls:
    for call in message.tool_calls:
        print('function_call:', call.function.name, call.function.arguments)
        call_output = make_call(call)
        messages.append(call_output)
else:
    print('ASSISTANT:')
    print(message.content)

function_call: search {"query":"How to join the course"}


In [65]:
messages

[{'role': 'system',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_RVOrjJBdDIuRMxLAgYLK6gbq', function=Function(arguments='{"query":"How to join the course"}', name='search'), type='function')]},
 {'type': 'tool',
  'tool_call_id': 'call_RVOrjJBdDIuRMxLAgYLK6gbq',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Rela

In [69]:
messages = [
    {'role': 'system', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.chat.completions.create(
        model='gpt-4o',
        messages=messages,
        tools=[search_tool]
    )

    messages.append({
    "role": "assistant",
    "content": message.content,
    "tool_calls": message.tool_calls
})


    if message.tool_calls:
        for call in message.tool_calls:
            print('function_call:', call.function.name, call.function.arguments)
            call_output = make_call(call)
            messages.append(call_output)
    else:
        print('ASSISTANT:')
    print(message.content)
    
    it = it + 1
    if has_function_calls == False:
            break

iteration #1...
function_call: search {"query":"How to join the course"}


In [71]:
def agent_loop(instructions, question, model='gpt-4o') -> str:
    messages = [
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1
    last_answer = ""

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        message = response.choices[0].message

        messages.append({
            "role": "assistant",
            "content": message.content,
            "tool_calls": message.tool_calls
        })

        if message.tool_calls:
            for call in message.tool_calls:
                print('function_call:', call.function.name, call.function.arguments)
                call_output = make_call(call)
                messages.append(call_output)
                has_function_calls = True
        else:
            print('ASSISTANT:')
            last_answer = message.content
            print(message.content)

        it = it + 1
        if not has_function_calls:
            break

    return last_answer

In [72]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [73]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"how to join this course"}
iteration #2...


RateLimitError: Error code: 429 - {'error': {'code': 'RateLimitReached', 'message': 'Rate limit of 50 per 86400s exceeded for UserByModelByDay. Please wait 691 seconds before retrying.', 'details': 'Rate limit of 50 per 86400s exceeded for UserByModelByDay. Please wait 691 seconds before retrying.'}}